<a href="https://colab.research.google.com/github/PioBasile/ProjetMachineLearning/blob/main/ProjetMachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from google.colab import drive
import pandas as pd
import re
import spacy
"""
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Téléchargement des ressources NLTK nécessaires
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
"""
# Avis de Reyder sur les algo de classification :
# K-NN : nul les tweet on des context different on ne peut pas les classer par similarité
# Naive Bayésiene : Bien, on a des classes précises et distincte
# Arbre décisionnel : Pas mal, mais il faudrait faire plein de sous classifications

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
path = '/content/drive/MyDrive/ColabNotebooks/scitweets_export.tsv'

# sep='\t' indique que le séparateur est une tabulation
df = pd.read_csv(path, sep='\t')

# 1 -- NETOYAGE TEXTE ------------

contractions_map = {
    ("don't", "dont"): "do not",
    ("doesn't", "doesnt"): "does not",
    ("can't", "cant"): "cannot",
    ("won't", "wont"): "will not",
    ("i'm", "im"): "i am",
    ("it's", "its"): "it is",
    ("you're", "youre"): "you are",
    ("they're", "theyre"): "they are",
    ("we're",): "we are",
    ("that's", "thats"): "that is",
    ("what's", "whats"): "what is"
}

def expand_contractions(text, cmap=contractions_map):
    flat_map = {k: v for keys, v in cmap.items() for k in keys}

    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []

    for t in tokens:
        key = t.lower()
        if key in flat_map:
            expanded.extend(flat_map[key].split())
        else:
            expanded.append(t)

    return " ".join(expanded)

def clean_tweet_text(text):
    if not isinstance(text, str):
        return ""

    # Suppression des URLs (liens http/https)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Suppression des mentions (@user)
    text = re.sub(r'@\w+', '', text)

    # Gestion des Hashtags (#science -> science)
    # On garde le mot mais on retire le symbole #
    text = re.sub(r'#(\w+)', r'\1', text)

    # Suppression des caractères spéciaux et emojis
    # Cette règle ne garde que les caractères alphanumériques et la ponctuation de base
    text = re.sub(r'[^\w\s\d\.\!\?\-\']', '', text)

    # Suppression des espaces en trop
    text = re.sub(r'\s+', ' ', text).strip()

    # Expansion des contractions
    text = expand_contractions(text)

    return text

df['text_cleaned'] = df['text'].apply(clean_tweet_text)

display(df[['text', 'text_cleaned']])

,text,text_cleaned
0,Knees are a bit sore. i guess that's a sign th...,Knees are a bit sore. i guess that is a sign t...
1,McDonald's breakfast stop then the gym 🏀💪,McDonald's breakfast stop then the gym
2,Can any Gynecologist with Cancer Experience ex...,Can any Gynecologist with Cancer Experience ex...
3,Couch-lock highs lead to sleeping in the couch...,Couch-lock highs lead to sleeping in the couch...
4,Does daily routine help prevent problems with ...,Does daily routine help prevent problems with ...
...,...,...
1135,@LaylaFanucci @realDonaldTrump I'm sorry but w...,i am sorry but we DO NOT have 14 of million de...
1136,"Dear #NIN applicants, you can kindly download ...",Dear NIN applicants you can kindly download th...
1137,Whats the uber support team email address?,what is the uber support team email address?
1138,House passes bill to increase stimulus checks ...,House passes bill to increase stimulus checks ...


In [25]:
"""
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def advanced_text_processing(text):
    if not isinstance(text, str):
        return ""

    # tokenisation
    words = text.lower().split()

    # élimination des stop_words
    cleaned_words =[]
    for word in words:
        if word not in stop_words:
            # lemmatisation
            lemma = lemmatizer.lemmatize(word)
            cleaned_words.append(lemma)

    return " ".join(cleaned_words)
"""
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

def advanced_text_processing(text):
    """Lemmatisation et suppression des Stop Words avec spaCy"""
    if not text:
        return ""

    # On passe le texte en minuscules à spaCy
    doc = nlp(text.lower())

    cleaned_words =[]
    for token in doc:
        # On exclut les stop words, la ponctuation et les espaces vides
        if not token.is_stop and not token.is_punct and not token.is_space:
            # On garde le lemme (token.lemma_)
            cleaned_words.append(token.lemma_)

    return " ".join(cleaned_words)

df['text_fully_cleaned'] = df['text_cleaned'].apply(advanced_text_processing)

display(df[['text_cleaned', 'text_fully_cleaned']])

,text_cleaned,text_fully_cleaned
0,Knees are a bit sore. i guess that is a sign t...,knee bit sore guess sign recent treadmilling work
1,McDonald's breakfast stop then the gym,mcdonald breakfast stop gym
2,Can any Gynecologist with Cancer Experience ex...,gynecologist cancer experience explain danger ...
3,Couch-lock highs lead to sleeping in the couch...,couch lock high lead sleep couch get to stop shit
4,Does daily routine help prevent problems with ...,daily routine help prevent problem bipolar dis...
...,...,...
1135,i am sorry but we DO NOT have 14 of million de...,sorry 14 million dead covid total death u.s ac...
1136,Dear NIN applicants you can kindly download th...,dear nin applicant kindly download enrolment f...
1137,what is the uber support team email address?,uber support team email address
1138,House passes bill to increase stimulus checks ...,house pass bill increase stimulus check 2000
